In [4]:
import pandas as pd

df = pd.read_csv(
    "/home/ubuntu/cur/program/Teaching/1/サンプルコード/1-3/MLB_pitch_data.csv",
    encoding="utf-8-sig",
)

In [5]:
df.head()

,pitches,player_id,player_name,total_pitches,pitch_type,pitch_percent,ba,iso,babip,slg,...,xslgdiff,wobadiff,swing_miss_percent,arm_angle,attack_angle,attack_direction,swing_path_tilt,rate_ideal_attack_angle,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,289,642547,"Peralta, Freddy",3086,FF,9.4,0.193,0.105,0.250,0.298,...,-0.086,-0.028,21.8,41.8,2.763975,11.595647,32.269736,0.415254,36.703650,20.904933
1,287,667755,"Soriano, José",2784,SI,10.3,0.240,0.080,0.246,0.320,...,-0.111,-0.055,15.6,36.4,2.745632,10.877299,35.275864,0.388060,37.017603,22.417583
2,282,669194,"Nelson, Ryne",2466,FF,11.4,0.194,0.222,0.196,0.417,...,-0.056,-0.047,14.0,55.5,3.407508,8.925722,30.271196,0.414966,34.847613,22.669756
3,267,592332,"Gausman, Kevin",3030,FF,8.8,0.237,0.102,0.283,0.339,...,0.007,0.007,15.9,37.3,3.180384,7.964539,34.909353,0.393939,36.079043,21.124763
4,265,675911,"Strider, Spencer",2110,FF,12.6,0.300,0.183,0.348,0.483,...,0.015,0.016,14.7,42.9,2.932559,10.443334,30.419934,0.354331,35.893245,22.256092


## 重回帰（`k_percent`）

- **目的変数**: `k_percent`
- **説明変数**: 投球物理量3つ（平均0・分散1に標準化）＋ `pitch_type` のダミー（出現回数20未満は `OTHER` にまとめる）
- **欠損**: 上記に必要な列で欠損のある行は削除
- **推定**: `statsmodels` の **WLS**（重み `pa`＝打席数が大きいほどレートの分散が小さいとみなす）

連続変数の係数は「1標準偏差増加に伴う `k_percent` の変化（ポイント）」の解釈になる。

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

cols = [
    "k_percent",
    "velocity",
    "spin_rate",
    "api_break_z_induced",
    "pitch_type",
    "pa",
]
d = df[cols].copy()
for c in cols:
    if c != "pitch_type":
        d[c] = pd.to_numeric(d[c], errors="coerce")

d = d.dropna().reset_index(drop=True)

rare_threshold = 20
counts = d["pitch_type"].value_counts()
rare_types = counts[counts < rare_threshold].index
d["pitch_type_g"] = np.where(d["pitch_type"].isin(rare_types), "OTHER", d["pitch_type"])

cont = ["velocity", "spin_rate", "api_break_z_induced"]
z_names = ["velocity_z", "spin_rate_z", "ivb_z"]
for raw, z in zip(cont, z_names):
    mu = d[raw].mean()
    sd = d[raw].std(ddof=0)
    d[z] = (d[raw] - mu) / sd

dummies = pd.get_dummies(d["pitch_type_g"], drop_first=True, dtype=float)
X = pd.concat([d[z_names], dummies], axis=1)
X = sm.add_constant(X, has_constant="add")
y = d["k_percent"]
w = d["pa"]

model = sm.WLS(y, X, weights=w).fit(cov_type="HC1")
print(model.summary())
print("\nOTHER にまとめた球種 (出現回数 < %d):" % rare_threshold)
print(sorted(rare_types.tolist()))